In [5]:
import ccxt
from datetime import datetime

# 1. 设置交易所 — 只能用 Coinbase（Colab 上其他交易所全被屏蔽）
exchange = ccxt.coinbase()

# 2. 获取BTC/USD和ETH/USD的订单簿深度（前10档）
def get_orderbook_depth(exchange, symbol, levels=10):
    """获取某个币种的前N档买卖深度"""
    ob = exchange.fetch_order_book(symbol, limit=levels)

    bid_depth = sum([vol for price, vol in ob['bids']])     # 买方总深度
    ask_depth = sum([vol for price, vol in ob['asks']])     # 卖方总深度
    best_bid = ob['bids'][0][0]                              # 最优买价
    best_ask = ob['asks'][0][0]                              # 最优卖价
    spread_bp = (best_ask - best_bid) / best_bid * 10000    # 价差（基点）

    return {
        'bid_depth': bid_depth,
        'ask_depth': ask_depth,
        'total_depth': bid_depth + ask_depth,
        'bid_ask_ratio': bid_depth / ask_depth if ask_depth > 0 else None,
        'spread_bp': spread_bp,
        'price': (best_bid + best_ask) / 2,
        'timestamp': datetime.now().strftime('%H:%M:%S')
    }

# 3. 同时拉BTC和ETH的订单簿作对比
symbols = ['BTC/USD', 'ETH/USD']

print(f"{'='*65}")
print(f"  BTC/ETH 订单簿深度对比 — Coinbase")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"{'='*65}")

for symbol in symbols:
    data = get_orderbook_depth(exchange, symbol)
    print(f"\n{'─'*40}")
    print(f"  📍 {symbol}")
    print(f"  当前价格: ${data['price']:,.2f}")
    print(f"  买方深度: {data['bid_depth']:.4f} {symbol.split('/')[0]}")
    print(f"  卖方深度: {data['ask_depth']:.4f} {symbol.split('/')[0]}")
    print(f"  总深度:   {data['total_depth']:.4f}")
    print(f"  买卖比:   {data['bid_ask_ratio']:.3f}")
    print(f"  价差:     {data['spread_bp']:.2f} bp")

    ratio = data['bid_ask_ratio']
    if ratio and ratio > 1.2:
        print(f"  信号: 🟢 买方强于卖方 (买卖比>1.2)")
    elif ratio and ratio < 0.8:
        print(f"  信号: 🔴 卖方强于买方 (买卖比<0.8)")
    else:
        print(f"  信号: ⚪ 多空均衡 (0.8<买卖比<1.2)")

# 4. 计算BTC/ETH深度比
btc_data = get_orderbook_depth(exchange, 'BTC/USD')
eth_data = get_orderbook_depth(exchange, 'ETH/USD')
btc_eth_depth_ratio = btc_data['total_depth'] / eth_data['total_depth']

print(f"\n{'='*65}")
print(f"  🔗 BTC Dominance 联动分析")
print(f"{'='*65}")
print(f"  BTC总深度: {btc_data['total_depth']:.4f} BTC")
print(f"  ETH总深度: {eth_data['total_depth']:.4f} ETH")
print(f"  BTC/ETH深度比: {btc_eth_depth_ratio:.2f}")
print(f"\n  解读:")
print(f"  • 深度比越高 → BTC流动性相对ETH越好")
print(f"  • 如果深度比下降但Dominance上升 → 异常信号")
print(f"    说明BTC Dominance是被动上升（山寨跌更惨），")
print(f"    而不是真金白银在买BTC")
print(f"{'='*65}")

  BTC/ETH 订单簿深度对比 — Coinbase
  2026-07-01 04:46

────────────────────────────────────────
  📍 BTC/USD
  当前价格: $59,331.51
  买方深度: 0.2689 BTC
  卖方深度: 1.1763 BTC
  总深度:   1.4452
  买卖比:   0.229
  价差:     0.00 bp
  信号: 🔴 卖方强于买方 (买卖比<0.8)

────────────────────────────────────────
  📍 ETH/USD
  当前价格: $1,599.20
  买方深度: 14.3624 ETH
  卖方深度: 56.4459 ETH
  总深度:   70.8083
  买卖比:   0.254
  价差:     0.06 bp
  信号: 🔴 卖方强于买方 (买卖比<0.8)

  🔗 BTC Dominance 联动分析
  BTC总深度: 1.4452 BTC
  ETH总深度: 70.8083 ETH
  BTC/ETH深度比: 0.02

  解读:
  • 深度比越高 → BTC流动性相对ETH越好
  • 如果深度比下降但Dominance上升 → 异常信号
    说明BTC Dominance是被动上升（山寨跌更惨），
    而不是真金白银在买BTC
